# Validación de la infraestructura (smoke test)

## Objetivo

Este notebook ejecuta un *smoke test* para validar la **disponibilidad operativa mínima** de la infraestructura desplegada con Docker desde JupyterLab.

El alcance de la prueba cubre solo checks rápidos de base:
- Conectividad y permisos de escritura/lectura en BD (`market_data`).
- Extensión TimescaleDB activa y presencia de `market_data.market_ohlcv` como hypertable.
- Conectividad con MLflow y registro/listado de un artefacto.
- Persistencia básica del bind mount de artefactos MLflow.

> **Rol de este notebook:** evidencia de salud mínima.
>
> **No cubre contrato completo de infraestructura** (esquemas/tablas críticas de todos los dominios y `feature_store`). Esa validación completa se realiza en `00_infraestructura/02_evidencia_infraestructura.ipynb`.

## Criterios de éxito

| # | Check | Tipo | Descripción |
|---|-------|------|-------------|
| 1 | **Conectividad BD (CRUD)** | Duro | Escribir, leer y borrar una tabla temporal en `market_data`. Si falla, el entorno no es operativo. |
| 2 | **Extensión TimescaleDB** | Duro | Confirmar que la extensión `timescaledb` está activa en `pg_extension`. |
| 3 | **Hypertable `market_ohlcv`** | Blando | Verificar que la tabla aparece como hypertable. Depende de inicialización previa. |
| 4 | **MLflow tracking + artefacto** | Duro | Registrar métricas y un artefacto; y confirmar que MLflow lo lista. |
| 5 | **Persistencia (volúmenes / bind mount MLflow)** | Duro | Verificar que `/app/models/mlflow` existe, es escribible y coherente con `artifact_uri`. |

> **Convención:** los checks **duros** detienen la ejecución con `AssertionError`; los **blandos** solo dejan warning.
>
> **Importante:** este criterio de éxito es de smoke mínimo. Se valida de forma completa en `02_evidencia_infraestructura.ipynb`.

## Prerrequisitos y ejecución

Antes de ejecutar este notebook, asegúrate de cumplir las siguientes condiciones:

1. **Servicios Docker activos:** ejecuta `docker compose up -d` desde la raíz del proyecto y espera *healthchecks* en `healthy`.
2. **Abrir desde `jupyter`:** este notebook debe ejecutarse en JupyterLab (`http://localhost:8888`) para usar variables inyectadas por Docker Compose.
3. **Ejecución completa:** `Kernel > Restart Kernel and Run All` (secuencial, sin saltar celdas).
4. **Complemento opcional (red externa):** tras un smoke local exitoso, `00_infraestructura/03_smoke_api_externa.ipynb` (opt-in `RUN_EXTERNAL_SMOKE=1`) valida conectividad HTTPS hacia APIs públicas; no sustituye este notebook.

**Variables de entorno requeridas**:

| Variable | Ejemplo | Propósito |
|----------|---------|-----------|
| `POSTGRES_USER` | `admin` | Usuario PostgreSQL |
| `POSTGRES_PASSWORD` | `***` | Contraseña PostgreSQL |
| `POSTGRES_HOST` | `db` | Host del servicio Docker |
| `POSTGRES_PORT` | `5432` | Puerto interno PostgreSQL |
| `POSTGRES_DB` | `tfg_db` | Base de datos |
| `MLFLOW_TRACKING_URI` | `http://mlflow:5000` | URI del tracking server |

In [1]:
import os
import sys
import tempfile
import time
import warnings
from pathlib import Path

# Silencia warning conocido de dependencias internas de MLflow.
warnings.filterwarnings(
    "ignore",
    message=".*pkg_resources is deprecated.*",
    category=UserWarning,
)

import mlflow
import pandas as pd
from mlflow.tracking import MlflowClient
from sqlalchemy import create_engine, text

# Incorpora la raíz del proyecto al path para resolver imports de `src` en notebook.
PROJECT_ROOT = Path.cwd().parents[1]
project_root_str = str(PROJECT_ROOT)
if project_root_str not in sys.path:
    sys.path.append(project_root_str)

from src.infrastructure import configure_logger, logger

configure_logger()
log = logger.bind(trabajo="validacion_smoke_test")

ENV_VARS = {
    "postgres_user": "POSTGRES_USER",
    "postgres_password": "POSTGRES_PASSWORD",
    "postgres_host": "POSTGRES_HOST",
    "postgres_db": "POSTGRES_DB",
    "postgres_port": "POSTGRES_PORT",
    "mlflow_tracking_uri": "MLFLOW_TRACKING_URI",
}

CHECK_DEFINITIONS = {
    "BD_CRUD": {
        "description": "Conectividad BD (escritura/lectura/borrado)",
        "type": "Duro",
    },
    "TIMESCALEDB_EXT": {
        "description": "Extensión TimescaleDB activa",
        "type": "Duro",
    },
    "HYPERTABLE": {
        "description": "Hypertable market_data.market_ohlcv",
        "type": "Blando",
    },
    "MLFLOW": {
        "description": "MLflow tracking + artefacto",
        "type": "Duro",
    },
    "PERSISTENCIA": {
        "description": "Volúmenes Docker (bind mounts)",
        "type": "Duro",
    },
}

SMOKE_SCHEMA = "market_data"
SMOKE_TABLE_NAME = "smoke_test_table"
SMOKE_TABLE = f"{SMOKE_SCHEMA}.{SMOKE_TABLE_NAME}"
MLFLOW_EXPERIMENT_NAME = "SMOKE_TEST_INFRA"
MLFLOW_RUN_NAME = "infra_validation_v1"
MLFLOW_ARTIFACT_DIR = Path("/app/models/mlflow")
MLFLOW_ARTIFACT_NAME = "smoke_artifact.txt"
MLFLOW_ARTIFACT_TEXT = "SMOKE_TEST_INFRA artifact OK\n"

config = {key: os.getenv(env_var) for key, env_var in ENV_VARS.items()}
missing_env = [ENV_VARS[key] for key, value in config.items() if not value]
assert not missing_env, (
    f"Variables de entorno requeridas no configuradas: {missing_env}. "
    "Verifica tu archivo .env y que el notebook se ejecuta desde el servicio 'jupyter'."
)

# Mantiene estado compartido del smoke test para construir resumen final
results: dict[str, str] = {}
artifact_uri: str | None = None


def set_check_result(check_id: str, status: str = "PASS") -> None:
    """Registra el resultado de un check esperado y emite un evento homogéneo."""
    assert check_id in CHECK_DEFINITIONS, f"Check no registrado en el contrato: {check_id}"
    results[check_id] = status
    log.info("check_superado", id_check=check_id, estado=status)


log.info(
    "chequeo_entorno",
    directorio_actual=os.getcwd(),
    version_mlflow=mlflow.__version__,
    uri_tracking=config["mlflow_tracking_uri"],
    variables_entorno_ok=True,
)

2026-06-11T12:35:00.596180Z [info     ] chequeo_entorno                directorio_actual=/app/notebooks/00_infraestructura trabajo=validacion_smoke_test uri_tracking=http://mlflow:5000 variables_entorno_ok=True version_mlflow=2.10.0


## Test de persistencia relacional (TimescaleDB/PostgreSQL)

A continuación, probaremos la conexión utilizando `SQLAlchemy`. El objetivo es:
1. Conectar a la BD usando las credenciales inyectadas por Docker (`.env`).
2. Crear/reescribir una tabla temporal (`smoke_test_table`) en el esquema `market_data`.
3. Leer los datos para confirmar el ciclo de escritura/lectura.
4. Borrar la tabla para dejar el entorno limpio.

> Nota: esta prueba valida conectividad y permisos (CRUD). La validación específica de Timescale (extensión/hypertable) se realiza en el siguiente bloque.

In [2]:
# Check 1: Valida conectividad BD mediante ciclo CRUD efímero sobre esquema técnico
db_url = (
    "postgresql+psycopg2://"
    f"{config['postgres_user']}:{config['postgres_password']}"
    f"@{config['postgres_host']}:{config['postgres_port']}/{config['postgres_db']}"
)
engine = create_engine(db_url)

crud_sample = pd.DataFrame(
    {"test_val": [1, 2, 3], "status": ["Alpha", "Beta", "Gamma"]}
)

crud_sample.to_sql(
    SMOKE_TABLE_NAME,
    engine,
    schema=SMOKE_SCHEMA,
    if_exists="replace",
    index=False,
)
log.info("escritura_bd_exitosa", tabla=SMOKE_TABLE, filas=len(crud_sample))

crud_readback = pd.read_sql(f"SELECT * FROM {SMOKE_TABLE}", engine)
assert len(crud_readback) == len(crud_sample), (
    f"CRUD read mismatch: escritas {len(crud_sample)} filas, "
    f"recuperadas {len(crud_readback)}."
)
log.info("lectura_bd_exitosa", filas_recuperadas=len(crud_readback))

with engine.connect() as connection:
    connection.execute(text(f"DROP TABLE IF EXISTS {SMOKE_TABLE}"))
    connection.commit()
log.info("limpieza_bd_exitosa", tabla=SMOKE_TABLE)

set_check_result("BD_CRUD")

2026-06-11T12:35:00.845416Z [info     ] escritura_bd_exitosa           filas=3 tabla=market_data.smoke_test_table trabajo=validacion_smoke_test
2026-06-11T12:35:00.853221Z [info     ] lectura_bd_exitosa             filas_recuperadas=3 trabajo=validacion_smoke_test
2026-06-11T12:35:00.866278Z [info     ] limpieza_bd_exitosa            tabla=market_data.smoke_test_table trabajo=validacion_smoke_test
2026-06-11T12:35:00.867274Z [info     ] check_superado                 estado=PASS id_check=BD_CRUD trabajo=validacion_smoke_test


## Validación semántica de TimescaleDB



Este bloque comprueba que el motor está inicializado como TSDB, no solo que responde:

- **Extensión activa**: verifica que `timescaledb` está cargada en `pg_extension`.
- **Hypertable disponible**: verifica que `market_data.market_ohlcv` aparece en `timescaledb_information.hypertables`.

Si la hypertable no aparece, se registra un error crítico e igualmente se informa si la tabla existe al menos como relación estándar.

In [3]:
# Check 2: Extensión TimescaleDB activa
timescaledb_extension = pd.read_sql(
    "SELECT extname FROM pg_extension WHERE extname = 'timescaledb';",
    engine,
)
assert not timescaledb_extension.empty, (
    "La extensión 'timescaledb' NO está instalada en pg_extension. "
    "Verifica que la imagen Docker es timescale/timescaledb y que init.sql ejecutó CREATE EXTENSION."
)
log.info("extension_timescaledb_ok", extension=timescaledb_extension["extname"].iloc[0])
set_check_result("TIMESCALEDB_EXT")

# Check 3: Valida hypertable esperada en dominio de mercado
market_ohlcv_hypertable = pd.read_sql(
    """
    SELECT hypertable_schema, hypertable_name
    FROM timescaledb_information.hypertables
    WHERE hypertable_schema = 'market_data'
      AND hypertable_name   = 'market_ohlcv';
    """,
    engine,
)

if market_ohlcv_hypertable.empty:
    market_ohlcv_relation = pd.read_sql(
        "SELECT to_regclass('market_data.market_ohlcv') AS regclass;",
        engine,
    )
    table_exists = market_ohlcv_relation["regclass"].iloc[0] is not None
    log.warning(
        "market_ohlcv_no_es_hypertable",
        tabla_existe=table_exists,
        pista=(
            "La tabla existe pero no es hypertable. Posible problema con init.sql."
            if table_exists
            else "La tabla no existe. Verifica que init.sql se ejecutó correctamente al crear el volumen."
        ),
    )
    set_check_result("HYPERTABLE", "WARN")
else:
    log.info(
        "market_ohlcv_hypertable_ok",
        esquema=market_ohlcv_hypertable["hypertable_schema"].iloc[0],
        tabla=market_ohlcv_hypertable["hypertable_name"].iloc[0],
    )
    set_check_result("HYPERTABLE")

2026-06-11T12:35:00.894267Z [info     ] extension_timescaledb_ok       extension=timescaledb trabajo=validacion_smoke_test
2026-06-11T12:35:00.895261Z [info     ] check_superado                 estado=PASS id_check=TIMESCALEDB_EXT trabajo=validacion_smoke_test
2026-06-11T12:35:00.919833Z [info     ] market_ohlcv_hypertable_ok     esquema=market_data tabla=market_ohlcv trabajo=validacion_smoke_test
2026-06-11T12:35:00.920616Z [info     ] check_superado                 estado=PASS id_check=HYPERTABLE trabajo=validacion_smoke_test


## Test de MLOps

Validaremos que el servicio de desarrollo (`jupyter`) puede comunicarse con el servidor de tracking (`mlflow`) a través de la red interna de Docker.

- **URI de tracking**: leída de la variable de entorno `MLFLOW_TRACKING_URI`.
- **Prueba**:
  - Creación/selección del experimento `SMOKE_TEST_INFRA`.
  - Registro de parámetros y métricas.
  - Registro de un artefacto (fichero de texto) y verificación de que MLflow lo lista para el `run_id`.

In [4]:
# Check 4: Valida end-to-end tracking y artefactos en MLflow
mlflow.set_tracking_uri(config["mlflow_tracking_uri"])
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
client = MlflowClient()

with mlflow.start_run(run_name=MLFLOW_RUN_NAME) as run:
    run_id = run.info.run_id

    mlflow.log_param("model_type", "smoke_test")
    mlflow.log_param("version", "1.0.0")

    tracking_start = time.time()
    mlflow.log_metric("dummy_metric", 1.0)
    latency_ms = round((time.time() - tracking_start) * 1000, 1)
    mlflow.log_metric("tracking_latency_ms", latency_ms)

    with tempfile.TemporaryDirectory() as tmp_dir:
        artifact_path = Path(tmp_dir) / MLFLOW_ARTIFACT_NAME
        artifact_path.write_text(MLFLOW_ARTIFACT_TEXT, encoding="utf-8")
        mlflow.log_artifact(str(artifact_path))

    listed_artifacts = client.list_artifacts(run_id)
    listed_artifact_names = [artifact.path for artifact in listed_artifacts]
    assert MLFLOW_ARTIFACT_NAME in listed_artifact_names, (
        f"El artefacto '{MLFLOW_ARTIFACT_NAME}' no aparece en la lista del servidor. "
        f"Artefactos listados: {listed_artifact_names}"
    )

    artifact_uri = client.get_run(run_id).info.artifact_uri

    log.info(
        "validacion_mlflow_exitosa",
        experimento=MLFLOW_EXPERIMENT_NAME,
        id_run=run_id,
        artefacto_ok=True,
        artefactos=listed_artifact_names,
        uri_artefacto=artifact_uri,
        latencia_tracking_ms=latency_ms,
    )

set_check_result("MLFLOW")

2026-06-11T12:35:02.962170Z [info     ] validacion_mlflow_exitosa      artefacto_ok=True artefactos=['smoke_artifact.txt'] experimento=SMOKE_TEST_INFRA id_run=0004422d2abd428082f42b2e0c2ce610 latencia_tracking_ms=41.3 trabajo=validacion_smoke_test uri_artefacto=/app/models/mlflow/4/0004422d2abd428082f42b2e0c2ce610/artifacts
2026-06-11T12:35:02.992478Z [info     ] check_superado                 estado=PASS id_check=MLFLOW trabajo=validacion_smoke_test


## Validación de persistencia (volúmenes Docker)


Este bloque verifica que los volúmenes Docker están mapeados correctamente y que los directorios de persistencia son accesibles y escribibles desde el servicio `jupyter`.

| Volumen | Host | Contenedor | Propósito |
|---------|------|-----------|-----------|
| `postgres_data` | Docker named volume | `/var/lib/postgresql/data` | Datos PostgreSQL/TimescaleDB |
| `./models/mlflow` | Bind mount | `/app/models/mlflow` | Artefactos MLflow (modelos, samples) |

> **Nota:** El volumen de PostgreSQL (`postgres_data`) es un *named volume* gestionado por Docker y no es directamente accesible desde `jupyter`. Su persistencia queda validada implícitamente por el check de BD (CRUD). Para los artefactos de MLflow, verificamos acceso directo al bind mount y coherencia con el `artifact_uri` del run registrado.

In [5]:
# Check 5: Valida persistencia de volúmenes Docker
assert MLFLOW_ARTIFACT_DIR.exists(), (
    f"El directorio de artefactos '{MLFLOW_ARTIFACT_DIR}' no existe. "
    "Verifica el bind mount en docker-compose.yml: ./models/mlflow:/app/models/mlflow"
)

write_check_path = MLFLOW_ARTIFACT_DIR / ".smoke_test_write_check"
try:
    write_check_path.write_text("write_check_ok", encoding="utf-8")
    assert write_check_path.read_text(encoding="utf-8") == "write_check_ok"
    write_check_path.unlink()
    log.info("directorio_artefactos_mlflow_escribible", ruta=str(MLFLOW_ARTIFACT_DIR))
except PermissionError:
    raise AssertionError(
        f"Sin permisos de escritura en '{MLFLOW_ARTIFACT_DIR}'. "
        "Verifica los permisos del bind mount y la directiva 'user' en docker-compose.yml."
    )

if artifact_uri:
    assert artifact_uri.startswith(str(MLFLOW_ARTIFACT_DIR)), (
        f"El artifact_uri de MLflow ({artifact_uri}) no apunta al directorio de persistencia "
        f"esperado ({MLFLOW_ARTIFACT_DIR}). Los artefactos podrían no persistir entre reinicios."
    )
    log.info(
        "persistencia_uri_artefacto_ok",
        uri_artefacto=artifact_uri,
        montaje=str(MLFLOW_ARTIFACT_DIR),
    )
else:
    log.warning(
        "check_uri_artefacto_omitido",
        motivo="El check de MLflow no se ejecutó previamente",
    )

log.info(
    "validacion_persistencia_ok",
    artefactos_mlflow=str(MLFLOW_ARTIFACT_DIR),
    volumen_bd="postgres_data (named volume, verificado indirectamente vía CRUD)",
)

set_check_result("PERSISTENCIA")

2026-06-11T12:35:03.011664Z [info     ] directorio_artefactos_mlflow_escribible ruta=/app/models/mlflow trabajo=validacion_smoke_test
2026-06-11T12:35:03.012551Z [info     ] persistencia_uri_artefacto_ok  montaje=/app/models/mlflow trabajo=validacion_smoke_test uri_artefacto=/app/models/mlflow/4/0004422d2abd428082f42b2e0c2ce610/artifacts
2026-06-11T12:35:03.013424Z [info     ] validacion_persistencia_ok     artefactos_mlflow=/app/models/mlflow trabajo=validacion_smoke_test volumen_bd=postgres_data (named volume, verificado indirectamente vía CRUD)
2026-06-11T12:35:03.014223Z [info     ] check_superado                 estado=PASS id_check=PERSISTENCIA trabajo=validacion_smoke_test


## Resumen de validación

### Artefactos bajo `reports/`

Este notebook **no** exporta figuras ni tablas a `reports/`. Los artefactos de MLflow quedan bajo `/app/models/mlflow` (bind mount) y en el tracking server. La evidencia contractual persistida para memoria vive en `00_infraestructura/02_evidencia_infraestructura.ipynb` (`reports/validation/infra/`).

In [6]:
# Genera resumen tabular de checks esperados y estado final de smoke test
summary_rows = []
all_hard_pass = True

for check_id, definition in CHECK_DEFINITIONS.items():
    status = results.get(check_id, "FAIL")
    is_hard_check = definition["type"] == "Duro"
    if is_hard_check and status != "PASS":
        all_hard_pass = False

    summary_rows.append(
        {
            "Check": check_id,
            "Descripción": definition["description"],
            "Tipo": definition["type"],
            "Resultado": status,
        }
    )

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

if all_hard_pass:
    log.info("smoke_test_superada", resultados=results)
    print("=" * 60)
    print("  SMOKE TEST SUPERADO: Infraestructura operativa.")
    print("=" * 60)
else:
    log.critical("smoke_test_fallida", resultados=results)
    print("=" * 60)
    print("  SMOKE TEST FALLIDO: Revisar los checks marcados FAIL.")
    print("=" * 60)

try:
    engine.dispose()
    log.info("motor_bd_liberado")
except Exception:
    pass

,Check,Descripción,Tipo,Resultado
0,BD_CRUD,Conectividad BD (escritura/lectura/borrado),Duro,PASS
1,TIMESCALEDB_EXT,Extensión TimescaleDB activa,Duro,PASS
2,HYPERTABLE,Hypertable market_data.market_ohlcv,Blando,PASS
3,MLFLOW,MLflow tracking + artefacto,Duro,PASS
4,PERSISTENCIA,Volúmenes Docker (bind mounts),Duro,PASS


2026-06-11T12:35:03.047906Z [info     ] smoke_test_superada            resultados={'BD_CRUD': 'PASS', 'TIMESCALEDB_EXT': 'PASS', 'HYPERTABLE': 'PASS', 'MLFLOW': 'PASS', 'PERSISTENCIA': 'PASS'} trabajo=validacion_smoke_test
  SMOKE TEST SUPERADO: Infraestructura operativa.
2026-06-11T12:35:03.049648Z [info     ] motor_bd_liberado              trabajo=validacion_smoke_test
